# Swin Transformer Audio Training - AF_V3 (Swish Activation)

**Dataset Required:** `bishertello/asvspoof-21-df-cqt/my_dataset`

**GPU:** T4 x2 Accelerator

In [1]:
import os
import subprocess
import sys

# ========== CONFIGURATION ==========
REPO_URL = "https://github.com/MuhammadQaiser1921/swin-model.git"
REPO_NAME = "swin-model"
REPO_BRANCH = "AF_V3"  # ← Updated to use your branch with Swish activation
REPO_PATH = f"/kaggle/working/{REPO_NAME}"

# ========== CLONE OR PULL REPO ==========
if not os.path.exists(REPO_PATH):
    print(f"📌 Cloning branch '{REPO_BRANCH}' from {REPO_URL}...")
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL], check=True)
else:
    print(f"📌 Repository exists. Fetching updates for branch '{REPO_BRANCH}'...")
    os.chdir(REPO_PATH)
    subprocess.run(["git", "reset", "--hard"], check=True)
    subprocess.run(["git", "fetch", "--all"], check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "pull", "origin", REPO_BRANCH], check=True)
    os.chdir("/kaggle/working")

# ========== SETUP PATHS & REQUIREMENTS ==========
sys.path.append(f"{REPO_PATH}/src")

req_file = f"{REPO_PATH}/requirements.txt"
if os.path.exists(req_file):
    print("Installing requirements...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req_file], check=True)

print(f"✅ Repository (Branch: {REPO_BRANCH}) is ready and paths are configured.")

📌 Cloning branch 'AF_V3' from https://github.com/MuhammadQaiser1921/swin-model.git...


Cloning into 'swin-model'...


Installing requirements...
✅ Repository (Branch: AF_V3) is ready and paths are configured.


In [2]:
import tensorflow as tf

# GPU Configuration
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print("✅ GPU detected:", len(physical_devices), "device(s)")
    for gpu in physical_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
        print(f"  - {gpu}")
else:
    print("⚠️ No GPU detected. Using CPU.")

print("TensorFlow Version:", tf.__version__)
print("Available devices:", tf.config.list_logical_devices())

2026-03-13 16:46:52.471901: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773420412.718360      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773420412.779452      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773420413.268855      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773420413.268910      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773420413.268914      55 computation_placer.cc:177] computation placer alr

✅ GPU detected: 1 device(s)
  - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
TensorFlow Version: 2.19.0
Available devices: [LogicalDevice(name='/device:CPU:0', device_type='CPU'), LogicalDevice(name='/device:GPU:0', device_type='GPU')]


I0000 00:00:1773420440.470998      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [3]:
# Import training functions
from train_audio import load_and_prepare_data, run_training_session, Config

print("✅ Training modules imported successfully")
print(f"Dataset path: {Config.DATA_ROOT}")
print("Using train_audio.run_training_session for consistent training/evaluation flow")

✅ Training modules imported successfully
Dataset path: /kaggle/input/datasets/bishertello/asvspoof-21-df-cqt/my_dataset
Using train_audio.run_training_session for consistent training/evaluation flow


In [4]:
# Load and validate data dictionary
print("📊 Loading audio dataset...")
data = load_and_prepare_data()

if not data:
    raise RuntimeError("❌ Failed to load data. Check Kaggle dataset attachment and paths.")

print("✅ Data loaded successfully")
print(f"Train samples: {len(data['train_paths'])}")
print(f"Val samples: {len(data['val_paths'])}")
print(f"Test samples: {len(data['test_paths'])}")

📊 Loading audio dataset...
📂 Loading audio CQT dataset paths...
Train samples: 59325
Val samples: 18576
Test samples: 533927
✅ Data loaded successfully
Train samples: 59325
Val samples: 18576
Test samples: 533927


In [5]:
# Train using repository pipeline
from datetime import datetime

print(f"🚀 Starting training for {Config.epochs} epochs...")
print(f"Batch size: {Config.batch_size}")
print(f"Learning rate: {Config.lr}")
print("Activation function variant: AF_V3 (Swish in MLP)")

start_time = datetime.now()
model, history, test_metrics = run_training_session(
    data=data,
    epochs=Config.epochs,
    batch_size=Config.batch_size,
    lr=Config.lr
)
end_time = datetime.now()
training_duration = end_time - start_time

print(f"✅ Training completed in {training_duration}")

🚀 Starting training for 3 epochs...
Batch size: 16
Learning rate: 0.0001
Activation function variant: AF_V3 (Swish in MLP)
🚀 Training audio model for 3 epochs...
Epoch 1/3


I0000 00:00:1773422169.151372     138 service.cc:152] XLA service 0x7fcef4010770 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773422169.151415     138 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1773422174.498584     138 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1773422195.851809     138 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


3708/3708 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.9079 - loss: 0.2342
💾 Saved best .pth at epoch 1 (val_accuracy: 0.8934) -> /kaggle/working/models/audio_checkpoints/audio_best_model_20260313_171542.pth
3708/3708 ━━━━━━━━━━━━━━━━━━━━ 550s 134ms/step - accuracy: 0.9080 - loss: 0.2342 - val_accuracy: 0.8934 - val_loss: 1.0787
Epoch 2/3
3708/3708 ━━━━━━━━━━━━━━━━━━━━ 464s 125ms/step - accuracy: 0.9506 - loss: 0.1308 - val_accuracy: 0.8932 - val_loss: 1.6403
Epoch 3/3
3708/3708 ━━━━━━━━━━━━━━━━━━━━ 465s 125ms/step - accuracy: 0.9623 - loss: 0.1038 - val_accuracy: 0.8934 - val_loss: 2.1449
🧪 Evaluating on test split...
33371/33371 ━━━━━━━━━━━━━━━━━━━━ 1388s 42ms/step - accuracy: 0.8999 - loss: 1.0719
Test metrics: {'accuracy': 0.9768357872962952, 'loss': 0.23769108951091766}

📌 Threshold metrics @ 0.50
Confusion Matrix [[TN, FP], [FN, TP]]:
[[  3379  11490]
 [   878 518180]]
Precision: 0.9783 | Recall: 0.9983 | F1: 0.9882 | Accuracy: 0.9768
✅ Training completed in 1:11:47.253618


In [6]:
# Summarize training metrics
final_train_acc = float(history.history['accuracy'][-1])
final_train_loss = float(history.history['loss'][-1])
final_val_acc = float(history.history['val_accuracy'][-1])
final_val_loss = float(history.history['val_loss'][-1])

print("\n📊 AF_V3 Audio Model Results (Swish Activation)")
print("=" * 50)
print(f"accuracy: {final_train_acc:.4f}")
print(f"loss: {final_train_loss:.4f}")
print(f"val_accuracy: {final_val_acc:.4f}")
print(f"val_loss: {final_val_loss:.4f}")
print(f"test_metrics: {test_metrics}")
print(f"training_duration: {training_duration}")
print("=" * 50)


📊 AF_V3 Audio Model Results (Swish Activation)
accuracy: 0.9811
loss: 0.0519
val_accuracy: 0.8934
val_loss: 2.1449
test_metrics: {'accuracy': 0.9768357872962952, 'loss': 0.23769108951091766, 'threshold_metrics': {'threshold': 0.5, 'tn': 3379, 'fp': 11490, 'fn': 878, 'tp': 518180, 'precision': 0.978307247909057, 'recall': 0.9983084741974693, 'f1': 0.9882066605991993, 'threshold_accuracy': 0.9768357846671923}}
training_duration: 1:11:47.253618


In [7]:
# Save results JSON for docx update
import json

audio_results = {
    'branch': 'AF_V3',
    'activation_function': 'MLP: Swish, Attention: Softmax, Output: Sigmoid',
    'accuracy': final_train_acc,
    'loss': final_train_loss,
    'val_accuracy': final_val_acc,
    'val_loss': final_val_loss,
    'test_metrics': test_metrics,
    'training_duration': str(training_duration),
    'epochs': Config.epochs,
    'batch_size': Config.batch_size,
    'learning_rate': Config.lr
}

with open('/kaggle/working/af_v3_audio_results.json', 'w') as f:
    json.dump(audio_results, f, indent=2)

print("💾 Saved: /kaggle/working/af_v3_audio_results.json")
print("Copy the printed metrics into Research_Results.docx")

💾 Saved: /kaggle/working/af_v3_audio_results.json
Copy the printed metrics into Research_Results.docx
